## Comparing VG deconstruct results to Centrolign all pairs induced results 

In [11]:
### Read in VG deconstruct results 

import pandas as pd

results = "/private/groups/patenlab/mira/centrolign/rGFA_tests/04_15_25_rGFA_vg_results/4b.per-sample-by-chrom.tsv"

vg_dc_df = pd.read_csv(results, sep="\t")
vg_dc_df.head()

,chrom,sample,variant_type,ref_context,count
0,chr10,HG00097,Deletion,Off-reference,0
1,chr10,HG00097,Insertion,Off-reference,0
2,chr10,HG00097,MNP,Off-reference,0
3,chr10,HG00097,SNP,Off-reference,1
4,chr10,HG00097,SV Deletion,Off-reference,0


In [3]:
### Read in induced pairwise per-cigar mutation rates 

import pandas as pd
from pathlib import Path

chromosomes = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]

output_dir = Path("/private/groups/patenlab/mira/centrolign/analysis/per_cigar_mutation_rates")

dfs = []
for chrom in chromosomes:
    fp = output_dir / f"output_{chrom}.tsv"
    if fp.exists():
        dfs.append(pd.read_csv(fp, sep="\t"))

induced_df = pd.concat(dfs, ignore_index=True)
induced_df

,sample1,sample2,chr,avg_array_len,aligned_bases,n_snvs,n_snvs_per_aligned_base,n_snvs_per_aligned_base_per_avg_len,n_short_indels,n_short_indels_per_aligned_base,n_short_indels_per_aligned_base_per_avg_len,n_svs,n_svs_per_avg_len
0,HG02040.1,NA18983.1,chr1,4054652.5,3245747,1649,0.000508,1.253004e-10,110,0.000034,8.358424e-12,290,0.000072
1,HG00099.2,HG04204.1,chr1,3743784.0,3062338,1944,0.000635,1.695635e-10,141,0.000046,1.229859e-11,294,0.000079
2,HG002.2,HG01252.1,chr1,3893989.5,3463767,1864,0.000538,1.381982e-10,177,0.000051,1.312290e-11,244,0.000063
3,HG01099.1,NA20805.1,chr1,6613613.0,6199955,1266,0.000204,3.087496e-11,50,0.000008,1.219390e-12,236,0.000036
4,HG03041.1,HG03516.1,chr1,3535938.5,2891810,828,0.000286,8.097592e-11,44,0.000015,4.303068e-12,140,0.000040
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4756,HG002.1,HG01261.1,chrY,308747.0,246302,327,0.001328,4.300085e-09,46,0.000187,6.049050e-10,15,0.000049
4757,HG00621.1,NA18945.1,chrY,304226.0,263387,31,0.000118,3.868753e-10,2,0.000008,2.495970e-11,4,0.000013
4758,CHM13.0,HG002.1,chrY,317335.5,317335,0,0.000000,0.000000e+00,1,0.000003,9.930323e-12,0,0.000000
4759,NA18952.1,NA19043.1,chrY,141483.5,106828,87,0.000814,5.756100e-09,6,0.000056,3.969724e-10,7,0.000049


In [4]:
# read in alpha sat array sizes

import pandas as pd
import os

# Path to the folder containing TSV files
folder_path = "/private/groups/migalab/juklucas/censat_regions/active_arrays"

# List all files in the folder
all_files = os.listdir(folder_path)

# Filter files matching asat_arrays_${chr}.tsv, exclude ones with "_raw"
tsv_files = [f for f in all_files if f.startswith("asat_arrays_") and f.endswith(".tsv") and "_raw" not in f]

all_dfs = []

for tsv_file in tsv_files:
    # Extract chr from filename using string split
    # Example: "asat_arrays_chr12.tsv" -> "chr12"
    chr_label = tsv_file.replace("asat_arrays_", "").replace(".tsv", "")

    # Build full path
    file_path = os.path.join(folder_path, tsv_file)

    # Read TSV
    df = pd.read_csv(file_path, sep="\t")

    # Keep only desired columns
    df = df[["asat_start", "asat_end", "sample_id", "haplotype"]].copy()

    # Create combined haplotype.assembly_id column
    df["sample"] = df["sample_id"].astype(str) + "." + df["haplotype"].astype(str)

    # Add chr column from filename
    df["chr"] = chr_label

    all_dfs.append(df)

# Concatenate all files into a single DataFrame
asat_df = pd.concat(all_dfs, ignore_index=True)

# Optional: keep only relevant columns
asat_df = asat_df[["asat_start", "asat_end", "sample", "chr"]]

print(asat_df.head())

   asat_start  asat_end     sample   chr
0    92070153  94778997  HG00097.1  chr2
1    92385520  94321652  HG00097.2  chr2
2    92130971  94547225  HG00099.1  chr2
3    92146667  94564932  HG00099.2  chr2
4    92601188  95044273  HG00126.1  chr2


In [ ]:
### Get diploid asat lengths 
asat_df["length"] = asat_df["asat_end"] - asat_df["asat_start"]
asat_df["sample_base"] = asat_df["sample"].str.rsplit(".", n=1).str[0]

asat_dip_df = (
    asat_df.groupby(["sample_base", "chr"], as_index=False)["length"]
    .sum()
    .rename(columns={"sample_base": "sample"})
)
# asat_dip_df[asat_dip_df['sample']=="HG00097"]

,sample,chr,length
24,HG00097,chr1,5375424
25,HG00097,chr10,4122084
26,HG00097,chr11,5781356
27,HG00097,chr12,5434254
28,HG00097,chr13,3333358
29,HG00097,chr14,5461909
30,HG00097,chr15,2005964
31,HG00097,chr16,5205398
32,HG00097,chr18,7810616
33,HG00097,chr19,3308940


In [ ]:
### For each sample, get the average number of variants 

# of rgfa variants per 

### Comparison of vg deconstruct to Jordan's variant matrix 

In [ ]:
- need to run script to parse the matrix and pull counts of variants per sample 
- check that I ran the matrix with indels included 